# EfficientNetB7 — TensorFlow / Keras

Compound-scaled EfficientNet: MBConv with depthwise conv (3×3 or 5×5), Squeeze-Excitation (se_ratio=0.25), and Swish activations.  
**Input: 600×600  |  ~66M params  |  dropout=0.5**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score

print("TF:", tf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _conv_bn(x, filters, kernel_size, strides=1, padding="same", activation="swish"):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding,
                      use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if activation:
        x = layers.Activation(activation)(x)
    return x


def _se_block(x, in_ch, se_ratio):
    """Squeeze-Excitation bottleneck sized by pre-expansion in_ch."""
    se_ch = max(1, int(in_ch * se_ratio))
    ch    = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, ch))(se)
    se = layers.Conv2D(se_ch, 1, activation="swish",   use_bias=True)(se)
    se = layers.Conv2D(ch,    1, activation="sigmoid", use_bias=True)(se)
    return layers.Multiply()([x, se])


def _mbconv(x, in_ch, out_ch, expand_ratio, stride, kernel_size, se_ratio=0.25):
    """MBConv: expand-1x1(opt) - DWConv-kxk - SE - project-1x1 - residual."""
    shortcut    = x
    expanded_ch = in_ch * expand_ratio

    # Expand (skipped when expand_ratio == 1)
    if expand_ratio != 1:
        x = _conv_bn(x, expanded_ch, 1)

    # Depthwise conv (kernel_size varies per stage: 3 or 5)
    x = layers.DepthwiseConv2D(kernel_size, strides=stride,
                               padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)

    # Squeeze-Excitation (se_ch based on pre-expansion in_ch)
    if se_ratio > 0:
        x = _se_block(x, in_ch, se_ratio)

    # Project (no activation)
    x = _conv_bn(x, out_ch, 1, activation=None)

    # Residual only when stride==1 and channels are unchanged
    if stride == 1 and in_ch == out_ch:
        x = layers.Add()([shortcut, x])

    return x


def _build_blocks(x, block_args):
    for stage in block_args:
        in_ch, out_ch = stage["in_ch"], stage["out_ch"]
        for i in range(stage["n"]):
            s  = stage["stride"] if i == 0 else 1
            ic = in_ch           if i == 0 else out_ch
            x  = _mbconv(x, ic, out_ch, stage["expand"],
                         s, stage["k"], se_ratio=stage["se"])
    return x

BLOCK_ARGS = [
    {"k":3, "n": 4, "in_ch": 64, "out_ch": 32, "expand":1, "stride":1, "se":0.25},
    {"k":3, "n": 7, "in_ch": 32, "out_ch": 48, "expand":6, "stride":2, "se":0.25},
    {"k":5, "n": 7, "in_ch": 48, "out_ch": 80, "expand":6, "stride":2, "se":0.25},
    {"k":3, "n":10, "in_ch": 80, "out_ch":160, "expand":6, "stride":2, "se":0.25},
    {"k":5, "n":10, "in_ch":160, "out_ch":224, "expand":6, "stride":1, "se":0.25},
    {"k":5, "n":13, "in_ch":224, "out_ch":384, "expand":6, "stride":2, "se":0.25},
    {"k":3, "n": 4, "in_ch":384, "out_ch":640, "expand":6, "stride":1, "se":0.25},
]


def build_efficientnet_b7(num_classes=1000, input_shape=(600, 600, 3)):
    """EfficientNet-B7: stem=64, head_conv=2560, input=600x600, dropout=0.5."""
    inputs = keras.Input(shape=input_shape)

    # Stem: 3x3 conv, stride 2
    x = _conv_bn(inputs, 64, 3, strides=2)

    # Seven MBConv stages (compound-scaled)
    x = _build_blocks(x, BLOCK_ARGS)

    # Head conv + GAP + Dropout + classifier
    x = _conv_bn(x, 2560, 1)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="EfficientNetB7")


if __name__ == "__main__":
    model = build_efficientnet_b7()
    model.summary()

In [ ]:
model = build_efficientnet_b7(num_classes=10)
model.summary()

In [ ]:
dummy = tf.random.normal([2, 600, 600, 3])
out   = model(dummy, training=False)
print("Output shape:", out.shape)

In [ ]:
total = sum(p.numpy().size for p in model.trainable_weights)
print(f"Trainable params: {total:,}")

In [ ]:
layer_info = [
    (l.name, l.__class__.__name__, l.count_params())
    for l in model.layers
]
for lname, typ, p in layer_info[-20:]:
    print(f"{lname:45s}  {typ:30s}  {p:>10,}")

## Training

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH     = 8
IMG_SIZE  = (600, 600)
TRAIN_DIR = "dataset/train"
VAL_DIR   = "dataset/val"

train_gen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 20,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip    = True,
    zoom_range         = 0.1,
)
val_gen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size = IMG_SIZE,
    batch_size  = 8,
    class_mode  = "categorical",
)
val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size = IMG_SIZE,
    batch_size  = 8,
    class_mode  = "categorical",
    shuffle     = False,
)

NUM_CLASSES = len(train_data.class_indices)
print("Classes:", NUM_CLASSES)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

model = build_efficientnet_b7(num_classes=NUM_CLASSES)
model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = "categorical_crossentropy",
    metrics   = ["accuracy"],
)

callbacks = [
    ModelCheckpoint(
        "efficientnet_b7_best.keras",
        monitor        = "val_accuracy",
        save_best_only = True,
        verbose        = 1,
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.1,
        patience = 5,
        min_lr   = 1e-7,
        verbose  = 1,
    ),
]

history = model.fit(
    train_data,
    validation_data = val_data,
    epochs          = 30,
    callbacks       = callbacks,
)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"],     label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Inference

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

CLASS_NAMES = list(train_data.class_indices.keys())


def predict_image(img_path, model, class_names, img_size=(600, 600)):
    img   = load_img(img_path, target_size=img_size)
    x     = img_to_array(img) / 255.0
    x     = np.expand_dims(x, axis=0)
    probs = model.predict(x, verbose=0)[0]
    idx   = np.argmax(probs)
    print(f"Prediction: {class_names[idx]}  ({probs[idx]*100:.1f}%)")
    return class_names[idx], probs


# predict_image("test.jpg", model, CLASS_NAMES)

## ROC-AUC

In [ ]:
val_gen.reset()
y_true = val_data.classes
y_prob = model.predict(val_data, verbose=1)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as auc_score

y_bin     = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
macro_auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
print(f"Macro ROC-AUC: {macro_auc:.4f}")

plt.figure(figsize=(8, 6))
for i in range(min(NUM_CLASSES, 10)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, lw=1.5,
             label=f"{CLASS_NAMES[i]} (AUC={auc_score(fpr, tpr):.2f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — EfficientNetB7")
plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()